In [1]:
import sys
sys.executable

'c:\\Users\\Carl Wheezer\\Documents\\git-test\\bart-ridership-dash\\.venv\\Scripts\\python.exe'

In [9]:
from pathlib import Path
import pandas as pd
import duckdb

In [10]:
pd.options.display.float_format = "{:,.0f}".format

In [11]:


PROJECT_ROOT = Path.cwd().parent
RIDERSHIP_SUMMARY = PROJECT_ROOT / "data" / "processed" / "station_ridership_monthly_summary.csv"

duckdb.sql(f"""
SELECT Year, Month, SUM(Ridership) AS total_ridership
FROM read_csv_auto('{RIDERSHIP_SUMMARY.as_posix()}')
GROUP BY Year, Month
ORDER BY Year, Month
""").df()


,Year,Month,total_ridership
0,2018,1,"395,222"
1,2018,2,"9,189,913"
2,2018,3,"10,278,960"
3,2018,4,"9,950,264"
4,2018,5,"10,486,845"
...,...,...,...
91,2025,8,"4,923,124"
92,2025,9,"5,047,121"
93,2025,10,"5,346,891"
94,2025,11,"4,409,065"


In [36]:
jan_18 = PROJECT_ROOT / "data" / "raw" / "ridership_2018" / "Ridership_201801.xlsx"
jan_18

WindowsPath('c:/Users/Carl Wheezer/Documents/git-test/bart-ridership-dash/data/raw/ridership_2018/Ridership_201801.xlsx')

In [37]:
xls = pd.ExcelFile(jan_18)
xls.sheet_names

['Weekday OD', 'Saturday OD', 'Sunday OD']

In [38]:
raw_weekday = pd.read_excel(jan_18, sheet_name="Weekday OD", header=None)
raw_weekday.shape 

(49, 48)

In [39]:
raw_weekday.head(10)

,0,1,2,3,4,5,6,7,8,9,...,38,39,40,41,42,43,44,45,46,47
0,Exit stations,Entry stations->,NaN,WEEKDAY,NaN,NaN,2018-01-01 00:00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,RM,EN,EP,NB,BK,AS,MA,19,12,...,NC,WP,SS,SB,SO,MB,WD,OA,WS,Exits
2,RM,13,109,82,65,366,102,138,149,180,...,2,35,12,19,59,27,8,12,7,"3,935"
3,EN,132,20,91,96,826,139,262,471,562,...,6,27,34,43,114,71,13,31,26,"8,247"
4,EP,86,81,13,47,659,83,134,298,336,...,4,11,10,11,60,32,7,25,11,"4,702"
5,NB,72,87,51,16,200,61,91,285,249,...,3,16,8,10,91,36,8,31,17,"4,396"
6,BK,406,897,668,208,36,361,362,562,528,...,49,113,34,39,180,139,49,74,78,"11,123"
7,AS,105,128,83,61,329,15,83,264,252,...,7,28,13,14,64,42,10,23,22,"5,086"
8,MA,154,268,121,87,351,84,42,196,228,...,63,260,31,31,109,68,27,27,34,"8,692"
9,19,163,478,302,291,513,262,190,30,59,...,173,281,69,72,89,135,106,35,110,"12,962"


In [40]:
raw_weekday.tail()

,0,1,2,3,4,5,6,7,8,9,...,38,39,40,41,42,43,44,45,46,47
44,MB,30,72,30,38,132,38,70,134,141,...,12,33,66,34,262,39,4,6,1,"5,924"
45,WD,5,15,8,9,48,12,28,115,121,...,1,5,8,5,41,5,15,28,5,"3,482"
46,OA,11,29,21,30,43,18,20,35,40,...,8,27,2,3,16,4,25,4,6,"1,074"
47,WS,8,30,11,15,87,23,35,110,91,...,3,23,2,2,16,1,5,8,26,"3,042"
48,Entries,"4,161","7,922","4,631","4,220","10,312","5,160","8,629","13,054","13,112",...,"2,703","6,379","3,500","3,460","5,711","6,262","3,346","1,222","3,135","395,222"


In [41]:
raw_weekday.iloc[0:3, 0:12]

,0,1,2,3,4,5,6,7,8,9,10,11
0,Exit stations,Entry stations->,NaN,WEEKDAY,NaN,NaN,2018-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN
1,NaN,RM,EN,EP,NB,BK,AS,MA,19,12,LM,FV
2,RM,13,109,82,65,366,102,138,149,180,38,85


In [45]:
def summarize_sheet(sheet_name):
    raw = pd.read_excel(jan_18, sheet_name=sheet_name, header=None)

    entry_stations = raw.iloc[1, 1:].tolist()
    exit_stations = raw.iloc[2:, 0].tolist()

    matrix = raw.iloc[2:, 1:].copy()
    matrix.insert(0, "Exit Station", exit_stations)
    matrix = matrix[matrix["Exit Station"] != "Entries"]

    matrix.columns = ["Exit Station"] + entry_stations

    station_cols = [col for col in matrix.columns[1:] if col != "Exits"]
    matrix[station_cols] = matrix[station_cols].apply(pd.to_numeric, errors="coerce")

    station_entries = matrix[station_cols].sum(axis=0).reset_index()
    station_entries.columns = ["Entry Station", "Ridership"]
    print(matrix["Exit Station"].tail())
    print(matrix["Exit Station"].eq("Entries").sum())
    return station_entries

weekday = summarize_sheet("Weekday OD")
saturday = summarize_sheet("Saturday OD")
sunday = summarize_sheet("Sunday OD")

weekday["Ridership"].sum(), saturday["Ridership"].sum(), sunday["Ridership"].sum()


43    SO
44    MB
45    WD
46    OA
47    WS
Name: Exit Station, dtype: object
0
43    SO
44    MB
45    WD
46    OA
47    WS
Name: Exit Station, dtype: object
0
43    SO
44    MB
45    WD
46    OA
47    WS
Name: Exit Station, dtype: object
0


(np.float64(395222.0952380953), np.float64(176529.5), np.float64(103732.25))

In [26]:
feb_18 = PROJECT_ROOT / "data" / "raw" / "ridership_2018" / "Ridership_201802.xlsx"
feb_18

WindowsPath('c:/Users/Carl Wheezer/Documents/git-test/bart-ridership-dash/data/raw/ridership_2018/Ridership_201802.xlsx')

In [27]:
feb18_xls = pd.ExcelFile(feb_18)
feb18_xls.sheet_names

['Avg Weekday OD', 'Avg Saturday OD', 'Avg Sunday OD', 'Total Trips OD']

In [31]:
feb_18_df = pd.read_excel(feb_18, sheet_name='Avg Weekday OD', header=None).head()
feb_18_df

,0,1,2,3,4,5,6,7,8,9,...,38,39,40,41,42,43,44,45,46,47
0,Exit stations,Entry stations->,NaN,WEEKDAY,NaN,NaN,2018-02-01 00:00:00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,RM,EN,EP,NB,BK,AS,MA,19,12,...,NC,WP,SS,SB,SO,MB,WD,OA,WS,Exits
2,RM,16,115,91,72,427,104,140,160,190,...,2,36,12,19,49,28,7,11,8,"4,167"
3,EN,135,21,91,104,937,141,275,496,572,...,6,30,31,42,105,71,13,32,25,8588
4,EP,93,86,12,51,746,84,134,314,347,...,4,14,10,12,61,32,8,28,12,"4,963"


In [28]:
def summarize_sheet(sheet_name):
    rawfeb = pd.read_excel(feb_18, sheet_name=sheet_name, header=None)

    entry_stations = rawfeb.iloc[1, 1:].tolist()
    exit_stations = rawfeb.iloc[2:, 0].tolist()

    matrix = rawfeb.iloc[2:, 1:].copy()
    matrix.insert(0, "Exit Station", exit_stations)
    matrix = matrix[matrix["Exit Station"] != "Entries"]

    matrix.columns = ["Exit Station"] + entry_stations

    station_cols = matrix.columns[1:]
    matrix[station_cols] = matrix[station_cols].apply(pd.to_numeric, errors="coerce")

    station_entries = matrix[station_cols].sum(axis=0).reset_index()
    station_entries.columns = ["Entry Station", "Ridership"]

    return station_entries

weekday = summarize_sheet("Avg Weekday OD")
saturday = summarize_sheet("Avg Saturday OD")
sunday = summarize_sheet("Avg Sunday OD")

weekday["Ridership"].sum(), saturday["Ridership"].sum(), sunday["Ridership"].sum()


(np.float64(827898.7368421052), np.float64(357571.5), np.float64(220926.5))

In [49]:
date_hr_18 = PROJECT_ROOT / "data" / "raw" / "Hourly_OD" / "date-hour-soo-dest-2018.csv"
date_hr_18

WindowsPath('c:/Users/Carl Wheezer/Documents/git-test/bart-ridership-dash/data/raw/Hourly_OD/date-hour-soo-dest-2018.csv')

In [51]:
date_hr_18_df = pd.read_csv(date_hr_18, header=None)
date_hr_18_df.head()

,0,1,2,3,4
0,2018-01-01,0,12TH,12TH,3
1,2018-01-01,0,12TH,16TH,1
2,2018-01-01,0,12TH,BAYF,1
3,2018-01-01,0,12TH,CAST,3
4,2018-01-01,0,12TH,CIVC,2


In [53]:
date_hr_18_df.dtypes

0      str
1    int64
2      str
3      str
4    int64
dtype: object

In [56]:
date_hr_18_df.columns = ["Date", "Hour", "Origin", "Destination", "Exits"]
date_hr_18_df.head()

,Date,Hour,Origin,Destination,Exits
0,2018-01-01,0,12TH,12TH,3
1,2018-01-01,0,12TH,16TH,1
2,2018-01-01,0,12TH,BAYF,1
3,2018-01-01,0,12TH,CAST,3
4,2018-01-01,0,12TH,CIVC,2


In [59]:
jan_hr_df = date_hr_18_df[date_hr_18_df["Date"].str.contains('2018-01', na=False)]
jan_hr_df.tail()

,Date,Hour,Origin,Destination,Exits
830114,2018-01-31,23,WOAK,PITT,1
830115,2018-01-31,23,WOAK,POWL,6
830116,2018-01-31,23,WOAK,RICH,1
830117,2018-01-31,23,WOAK,SFIA,4
830118,2018-01-31,23,WOAK,WOAK,2


In [60]:
jan_total = jan_hr_df["Exits"].sum()
jan_total

np.int64(9785965)